In [1]:
from transformers import pipeline
# initializing the summarization pipeline with the BART model
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

c:\Users\susov\OneDrive\Desktop\Mood-Score-\AI_Bot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\susov\OneDrive\Desktop\Mood-Score-\AI_Bot\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\susov\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as a

In [2]:
# example counseling sessions with mood scores before and after the session
sessions = [
    {
        "input": "I feel very stressed and I can't focus on my studies.",
        "response": "I understand this feels overwhelming. Let's start with small tasks.",
        "mood_before": 0.2,
        "mood_after": 0.25
    },
    {
        "input": "I tried studying but I still feel distracted.",
        "response": "That's okay, progress takes time. Try 25-minute focused sessions.",
        "mood_before": 0.25,
        "mood_after": 0.3
    },
    {
        "input": "I feel a bit better after breaking tasks, but still anxious.",
        "response": "You're improving. Anxiety is normal, but you're taking control.",
        "mood_before": 0.3,
        "mood_after": 0.35
    },
    {
        "input": "I completed one chapter today, but I still doubt myself.",
        "response": "That's a big achievement! Try to acknowledge your progress.",
        "mood_before": 0.35,
        "mood_after": 0.45
    },
    {
        "input": "I feel slightly confident now, but exams still scare me.",
        "response": "Fear is natural, but you're preparing well. Keep going.",
        "mood_before": 0.45,
        "mood_after": 0.5
    },
    {
        "input": "I studied consistently today and felt productive.",
        "response": "That's amazing! Consistency is key to success.",
        "mood_before": 0.5,
        "mood_after": 0.6
    },
    {
        "input": "I feel more in control of my studies now.",
        "response": "That's a strong sign of progress. You're improving a lot.",
        "mood_before": 0.6,
        "mood_after": 0.7
    },
    {
        "input": "I helped a friend today and felt good about myself.",
        "response": "Helping others boosts confidence. You're growing emotionally too.",
        "mood_before": 0.7,
        "mood_after": 0.78
    },
    {
        "input": "I feel confident and less stressed about exams now.",
        "response": "You've come a long way. Your effort is paying off.",
        "mood_before": 0.78,
        "mood_after": 0.85
    },
    {
        "input": "I feel motivated and ready to face my exams.",
        "response": "That's excellent! Believe in yourself and keep going strong.",
        "mood_before": 0.85,
        "mood_after": 0.92
    }
]

In [ ]:
# Summarize each session's input and response into a concise summary
summaries = []

for session in sessions:
    full_text = session["input"] + " " + session["response"]

    summary = summarizer(
        full_text,
        max_length=60,
        min_length=20,
        do_sample=False
    )[0]['summary_text']

    summaries.append(summary)

print("Session Summaries:", summaries)

Your max_length is set to 60, but your input_length is only 28. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)
Your max_length is set to 60, but your input_length is only 26. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)
Your max_length is set to 60, but your input_length is only 29. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=14)
Your max_length is set to 60, but your input_length is only 26. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=13)
Your max

Session Summaries: ['"I feel very stressed and I can\'t focus on my studies. I understand this feels overwhelming. Let\'s start with small tasks," she says.', "I tried studying but I still feel distracted. That's okay, progress takes time. Try 25-minute focused sessions.", "Anxiety is normal, but you're taking control. I feel a bit better after breaking tasks, but still anxious. You're improving.", "I completed one chapter today, but I still doubt myself. That's a big achievement! Try to acknowledge your progress.", "I feel slightly confident now, but exams still scare me. Fear is natural, but you're preparing well. Keep going.", "I studied consistently today and felt productive. That's amazing! Consistency is key to success.", "I feel more in control of my studies now. That's a strong sign of progress. You're improving a lot.", "I helped a friend today and felt good about myself. Helping others boosts confidence. You're growing emotionally too.", "I feel confident and less stressed ab

In [ ]:
# Combine all summaries into one text for a final summary
combined_text = " ".join(summaries)

final_summary = summarizer(
    combined_text,
    max_length=120,
    min_length=40,
    do_sample=False
)[0]['summary_text']

print("Final Summary:", final_summary)

Final Summary: Anxiety is normal, but you're taking control. I feel slightly confident now, but exams still scare me. I helped a friend today and felt good about myself. Helping others boosts confidence. You're growing emotionally too.


In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

# retrieving API keys from environment variables
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

In [ ]:
# Initialize the Google Generative AI model
from langchain_google_genai import GoogleGenerativeAI,ChatGoogleGenerativeAI

BlogModel = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",   # ✅ stable & supported
    temperature=0.7,
    google_api_key=GEMINI_API_KEY
)

In [15]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import  create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
# Define the system prompt for the AI Counselor Story Generator
system_prompt = """
You are an AI Counselor Story Generator.

🎯 ROLE:
- Convert student counseling data into a motivational anonymous story
- Help other students learn and feel inspired
- Highlight emotional growth and improvement journey

📊 INPUT CONTEXT:
You will receive:
- Student counseling summary (multiple sessions combined)
- Mood progression (before → after)
- Emotional journey (stress → confidence, etc.)
- Key struggles (exam, breakup, family, etc.)

🚨 IMPORTANT RULES:
- NEVER include personal details (name, location, identity)
- Always keep the story completely anonymous
- Focus on learning, growth, and transformation
- If mood is NOT improved → DO NOT create story (return action: "skip")

📖 STORY STRUCTURE:
1. Start with struggle (low mood, problem)
2. Show emotional challenges
3. Show small improvements
4. Show final growth (confidence, motivation)
5. Keep it simple, human, and relatable

💡 ENDING (VERY IMPORTANT):
- Add 2–3 key lessons learned
- Add a final motivational line like:
  "What we learn from this story is..."

🎯 OUTPUT FORMAT (JSON ONLY):

{{
  "title": "Short inspiring title",
  "story": "Full anonymous story...",
  "key_learnings": [
    "Lesson 1",
    "Lesson 2",
    "Lesson 3"
  ],
  "final_message": "Motivational ending line",
  "improvement": true/false,
  "action": "create_blog / skip"
}}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Student Summary:\n{student_summary}")
])

In [17]:
chain = prompt | BlogModel


In [18]:
# Invoke the chain with the student summary
response = chain.invoke({
   "student_summary": final_summary
})

print(response.content)

```json
{
  "title": "Finding Strength Beyond the Exam Hall",
  "story": "There was a student who often felt a knot in their stomach, especially when thinking about exams. Anxiety was a familiar companion, making even the idea of tests feel overwhelming. Despite knowing that anxiety is a normal human experience, it often felt like an uphill battle to manage. \n\nHowever, something began to shift. This student started to actively work on taking control, understanding that while the fear of exams still lingered, it didn't have to define them. A turning point came when they had the chance to help a friend. In that moment of offering support and care, a surprising warmth spread through them. It wasn't about grades or performance; it was about connection and contribution. This simple act of kindness sparked a feeling of genuine goodness and competence.\n\nThat experience was a revelation: helping others had a powerful way of boosting their own confidence. Slowly but surely, a new sense of s